# Skyvonyx - train and measure your own detector

```
labelled scenes ──► tile at 1024px / 0.2 overlap ──► split BY SCENE ──► train YOLOv8 (resumable)
   (DOTA or COCO)    same slicer + stretch as the       train / val / test      checkpoints on Drive
                     inference pipeline
                                         ──► measure twice ──► publish to models/best.pt
                                              1. tile-level mAP (model.val)
                                              2. scene-level P/R/F1/AP50 through the WHOLE pipeline
```

Number 2 is the one you may quote to a customer, always with the test set described next to it.
Number 1 helps with training decisions only.

**One sensor, one class, one model to start with.** A ship detector for Sentinel-1 SAR and a building
detector for optical imagery are different models. Train the one your first pilot needs.

**SAR ship pilot path:** PRESET `sar_ship_hrsid` (or `sar_ship_ssdd`) to pretrain -> PRESET `sar_ship_own_s1`
with `BASE_WEIGHTS` = that run's `best.pt`, on your own labelled Sentinel-1 scenes -> score on the gold set
(`docs/GOLD_SET.md`). Measure after each step.

**Licences:** check every dataset's terms before training on it (`DATASET_LICENCE` below is written into the
model card). DOTA and xView are non-commercial; weights trained on them carry that restriction.

In [ ]:
#@title 1. Runtime + code (same as the worker notebook, cells 1-2)
import os, subprocess, sys, pathlib, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch to T4")
SKYVONYX_REPO = "https://github.com/harishrajCEO/skyvonyx.git"  #@param {type:"string"}
WORK = pathlib.Path("/content/skyvonyx")
def sh(c): print("$", c); subprocess.run(c, shell=True, check=True)
if not WORK.exists():
    sh(f"git clone --depth 1 {SKYVONYX_REPO} {WORK}")
VENDOR = WORK / "vendor"; VENDOR.mkdir(exist_ok=True)
if not (VENDOR / "sahi").exists():
    sh(f"git clone --depth 1 -q https://github.com/harishrajCEO/sahi.git {VENDOR/'sahi'}")
sh(f"pip install -q -r {WORK}/requirements-colab.txt")
sh(f"pip install -q -e {VENDOR}/sahi")
sys.path.insert(0, str(WORK)); sys.path.insert(0, str(VENDOR / "sahi"))
sh(f"cd {WORK} && python tests/test_training_eval.py | tail -1")

In [ ]:
#@title 2. Drive + what you are training (pick a PRESET; "custom" uses the fields below it)
from google.colab import drive; drive.mount("/content/drive")
from pathlib import Path
import re, json
from skyvonyx.config import SENSORS
ROOT = Path("/content/drive/MyDrive/skyvonyx")
D = "/content/drive/MyDrive/datasets"   # upload datasets here yourself; this notebook downloads nothing

PRESET   = "sar_ship_hrsid"   #@param ["sar_ship_hrsid", "sar_ship_ssdd", "sar_ship_own_s1", "custom"]
RUN_NAME = "ship_sar_v1"      #@param {type:"string"}

PRESETS = {
    # HRSID: 800x800 8-bit SAR chips with COCO json. After uploading, CHECK these paths match your folders.
    # Chip names like P0001_0_800_... encode the parent scene; GROUP_REGEX keeps each parent in one split.
    "sar_ship_hrsid": dict(
        LABEL_FORMAT="coco", IMAGES_DIR=f"{D}/HRSID/JPEGImages",
        LABELS=f"{D}/HRSID/annotations/train2017.json,{D}/HRSID/annotations/test2017.json",
        CLASSES="ship", SENSOR="sar_rendered", TILE=1024, GROUP_REGEX=r"^([^_]+)_",
        DATASET_IDS="hrsid", DATASET_SOURCE="HRSID, official splits pooled and re-split by parent scene"),
    # SSDD: small 8-bit SAR chips with Pascal-VOC xml. Names carry no parent-scene info, so neighbouring
    # chips can leak across splits - treat its test score as optimistic.
    "sar_ship_ssdd": dict(
        LABEL_FORMAT="voc", IMAGES_DIR=f"{D}/SSDD/JPEGImages", LABELS=f"{D}/SSDD/Annotations",
        CLASSES="ship", SENSOR="sar_rendered", TILE=1024, GROUP_REGEX="",
        DATASET_IDS="ssdd", DATASET_SOURCE="SSDD, VOC release (record which copy you downloaded)"),
    # Your own CVAT ship labels on Sentinel-1 GeoTIFFs (linear backscatter) = the pilot's real input.
    # Never include gold-set scenes here (docs/GOLD_SET.md).
    "sar_ship_own_s1": dict(
        LABEL_FORMAT="coco", IMAGES_DIR=f"{D}/s1_ships_train/images",
        LABELS=f"{D}/s1_ships_train/annotations.json",
        CLASSES="ship", SENSOR="sar", TILE=1024, GROUP_REGEX="",
        DATASET_IDS="sentinel-1,skyvonyx-labels", DATASET_SOURCE="Skyvonyx-labelled Sentinel-1 IW GRD"),
}
# ---- custom (used only when PRESET == "custom")
LABEL_FORMAT   = "dota"      #@param ["dota", "coco", "voc"]
IMAGES_DIR     = ""          #@param {type:"string"}
LABELS         = ""          #@param {type:"string"}
CLASSES        = "ship"      #@param {type:"string"}
SENSOR         = "optical"   #@param ["optical", "sar", "sar_rendered"]
TILE           = 1024        #@param {type:"integer"}
GROUP_REGEX    = ""          #@param {type:"string"}
DATASET_IDS    = ""          #@param {type:"string"}
DATASET_SOURCE = ""          #@param {type:"string"}
if PRESET != "custom":
    globals().update(PRESETS[PRESET])

classes = [c.strip() for c in CLASSES.split(",") if c.strip()]
dataset_ids = [d.strip() for d in DATASET_IDS.split(",") if d.strip()]
group_fn = None
if GROUP_REGEX:
    _rx = re.compile(GROUP_REGEX)
    def group_fn(stem):
        m = _rx.match(stem)
        return m.group(1) if m else stem
RUN_DIR = ROOT / "training_runs" / RUN_NAME; RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"preset={PRESET} classes={classes} sensor={SENSOR} tile={TILE} datasets={dataset_ids}")
print("sensor means:", SENSORS[SENSOR])

In [ ]:
#@title 3. Build the tiled dataset on local disk (Drive is too slow to train from)
import shutil
from skyvonyx import datasets

if LABEL_FORMAT == "dota":
    scenes = datasets.manifest_from_dota(IMAGES_DIR, LABELS)
elif LABEL_FORMAT == "voc":
    scenes = datasets.load_voc(LABELS, IMAGES_DIR)
else:
    scenes = [s for js in LABELS.split(",") for s in datasets.load_coco(js.strip(), IMAGES_DIR)]
missing = [s for s in scenes if not s.image.exists()]
scenes = [s for s in scenes if s.image.exists()]
print(len(scenes), "labelled images,", sum(len(s.boxes) for s in scenes), "boxes before class filter",
      f"({len(missing)} label entries had no image file - check IMAGES_DIR)" if missing else "")
if group_fn:
    print("parent-scene groups:", len({group_fn(s.scene_id) for s in scenes}),
          "e.g.", [group_fn(s.scene_id) for s in scenes[:3]], "<- should look like scene ids, not chip names")

DS = Path("/content/ds_" + RUN_NAME)
if DS.exists(): shutil.rmtree(DS)
card = datasets.build_dataset(
    scenes, DS, classes,
    spec=datasets.TileSpec(tile=TILE, overlap=0.2, sensor=SENSOR, min_visible=0.5,
                           negative_frac=0.1, pad_to_tile=True),
    source=DATASET_SOURCE, dataset_ids=dataset_ids, group_fn=group_fn)
for f in ("dataset_card.json", "test_scenes_gold.json"):
    shutil.copy(DS / f, RUN_DIR / f)
print(json.dumps(card["splits"], indent=1))
print("dropped (class not in list):", card["dropped_unknown_class"])
print("data licence:", "commercial-ok" if card["licence_status"]["commercial_ok"] else "NOT commercial-ok",
      card["licence_status"]["blocking"])
if card["splits"].get("test", {}).get("objects", 0) < 100:
    print("!! fewer than 100 test objects - scores will be too noisy to quote. Add labelled scenes.")

In [ ]:
#@title 4. Train (resumes automatically after a disconnect - just re-run cells 1-4)
from ultralytics import YOLO   # AGPL-3.0
BASE_WEIGHTS = "yolov8s.pt"   #@param {type:"string"}
# yolov8n.pt / yolov8s.pt / yolov8m.pt, OR a previous run's best.pt to fine-tune, e.g. the HRSID run:
# /content/drive/MyDrive/skyvonyx/training_runs/ship_sar_v1/train/weights/best.pt
EPOCHS = 50                   #@param {type:"integer"}
BATCH  = 8                    #@param {type:"integer"}
# T4 at 1024px: yolov8s batch 8 fits; drop to 4 on an out-of-memory error.

# Licence lineage: a fine-tune inherits every dataset its parent model saw.
inherited = []
if "/" in BASE_WEIGHTS:
    parent_card = Path(BASE_WEIGHTS).parents[2] / "model_card.json"
    if parent_card.exists():
        inherited = json.loads(parent_card.read_text()).get("dataset_ids", [])
        print("fine-tuning from", BASE_WEIGHTS, "- inherits datasets:", inherited)
    else:
        inherited = ["unknown-parent"]   # unknown lineage fails the licence check, on purpose
        print("!! no model_card.json beside", BASE_WEIGHTS, "- lineage unknown, cannot be commercial-ok")
all_dataset_ids = sorted(set(dataset_ids) | set(inherited))

last = RUN_DIR / "train" / "weights" / "last.pt"
if last.exists():
    print("resuming from", last)
    YOLO(str(last)).train(resume=True)
else:
    YOLO(BASE_WEIGHTS).train(
        data=str(DS / "data.yaml"), imgsz=TILE, epochs=EPOCHS, batch=BATCH,
        project=str(RUN_DIR), name="train", exist_ok=True,
        save_period=1,            # a checkpoint every epoch, on Drive
        patience=15, amp=True, workers=2, seed=0, deterministic=True,
        # overhead imagery has no "up": flips both ways are free augmentation.
        # SAR is grey: colour jitter would teach hues that never occur in it.
        fliplr=0.5, flipud=0.5, degrees=0.0, mosaic=1.0,
        **({"hsv_h": 0.0, "hsv_s": 0.0} if SENSOR.startswith("sar") else {}))
best = RUN_DIR / "train" / "weights" / "best.pt"
print("best:", best, best.exists())

In [ ]:
#@title 5. Measurement 1 - tile-level mAP on the held-out TEST split (for training decisions)
m = YOLO(str(best)).val(data=str(DS / "data.yaml"), split="test", imgsz=TILE, batch=BATCH, plots=False)
tile_metrics = {"mAP50": round(float(m.box.map50), 4), "mAP50_95": round(float(m.box.map), 4),
                "precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4),
                "per_class_ap50": {m.names[int(c)]: round(float(a), 4)
                                   for c, a in zip(m.box.ap_class_index, m.box.ap50)}}
(RUN_DIR / "tile_metrics_test.json").write_text(json.dumps(tile_metrics, indent=2))
print(json.dumps(tile_metrics, indent=2))

In [ ]:
#@title 6. Measurement 2 - the customer number: whole test SCENES through the real pipeline
# slicing + detection + SAHI re-stitch, scored against the full-scene gold. Duplicates the merge
# misses count as false positives here, exactly as they would in a delivery.
import tempfile
from skyvonyx.config import JobConfig
from skyvonyx.pipeline import run_job
from skyvonyx import evaluate

gold_path = RUN_DIR / "test_scenes_gold.json"
gold_coco = json.loads(gold_path.read_text())
job = Path(tempfile.mkdtemp()) / "eval_job"; (job / "input").mkdir(parents=True)
for im in gold_coco["images"]:
    shutil.copy(im["path"], job / "input" / Path(im["path"]).name)

CONF = 0.25   #@param {type:"number"}
cfg = JobConfig(job_id="eval_" + RUN_NAME, detector="yolov8", detector_weights=str(best),
                segmenter="none",           # boxes are what is scored; skip SAM 2 for speed
                tile=TILE, overlap=0.2, sensor=SENSOR, conf=0.01)   # keep low scores for AP
run_job(job, cfg, verbose=False)

result = evaluate.evaluate(evaluate.load_predictions(job / "output"),
                           evaluate.load_coco_boxes(gold_path), iou_thr=0.5, conf=CONF)
note = (f"{len(gold_coco['images'])} held-out scenes from {DATASET_SOURCE}, sensor={SENSOR}, "
        f"tile={TILE}. Split by scene hash, never seen in training.")
(RUN_DIR / "scene_accuracy_test.json").write_text(json.dumps(result, indent=2))
evaluate.write_accuracy_report(result, RUN_DIR / "ACCURACY_REPORT.md", note)
print(json.dumps(result["overall"], indent=2))
print(open(RUN_DIR / "ACCURACY_REPORT.md").read())

In [ ]:
#@title 7. Publish to the worker: models/best.pt + a model card (only if you accept the numbers)
import hashlib, datetime
from skyvonyx.datasets import licence_status
PUBLISH = False   #@param {type:"boolean"}
lic = licence_status(all_dataset_ids)
card_out = {"run": RUN_NAME, "created_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"),
            "base_weights": BASE_WEIGHTS, "weights_sha256_16": hashlib.sha256(best.read_bytes()).hexdigest()[:16],
            "classes": classes, "sensor": SENSOR, "tile": TILE, "overlap": 0.2,
            "dataset_ids": all_dataset_ids, "dataset_source": DATASET_SOURCE,
            "dataset": json.loads((RUN_DIR / "dataset_card.json").read_text())["splits"],
            "data_licence": lic, "commercial_ok": lic["commercial_ok"],
            "tile_metrics_test": tile_metrics, "scene_metrics_test": result["overall"],
            "scene_metrics_note": note,
            "detector_licence": "ultralytics YOLOv8 AGPL-3.0 (separate from data terms)",
            "use_only_on": f"imagery rendered as sensor={SENSOR}, similar resolution and region to the test set"}
(RUN_DIR / "model_card.json").write_text(json.dumps(card_out, indent=2))
print("commercial_ok:", lic["commercial_ok"], "|", lic["blocking"] or "no blockers")
if SENSOR == "sar_rendered":
    print("NOTE: trained on pre-rendered chips. On Sentinel-1 GeoTIFFs the worker must use sensor='sar',")
    print("      so the look differs from training. The S1 gold set (docs/GOLD_SET.md) measures that gap.")
    print("      To close it: PRESET sar_ship_own_s1 with BASE_WEIGHTS = this run's best.pt.")
if PUBLISH:
    models = ROOT / "models"; models.mkdir(exist_ok=True)
    if (models / "best.pt").exists():
        shutil.copy(models / "best.pt", models / f"best_before_{RUN_NAME}.pt")   # keep the old one
    shutil.copy(best, models / "best.pt")
    shutil.copy(RUN_DIR / "model_card.json", models / "best_model_card.json")
    print("published. The worker uses it from its next job (its cell 4 picks models/best.pt).")
    print("In job.json set tile =", TILE, "and sensor =", "'sar' for Sentinel-1 GeoTIFFs" if SENSOR.startswith("sar") else SENSOR)
else:
    print("model card written; tick PUBLISH and re-run to hand this model to the worker")